# Phase 3A: GTFS Exploratory Data Analysis

This notebook describes the scheduled routes, stop connectivity, trip templates, stop-time events, and recurring calendar patterns in the processed TGSRTC and HMRL GTFS feeds. Every count and plot is GTFS-derived scheduled service.

GTFS does **not** establish passenger demand, ridership, actual usage, or passenger volume. Route and stop connectivity are not passenger counts. No forecasting or machine-learning models are fitted.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

project_root = Path.cwd()
if not (project_root / 'src').is_dir():
    if (project_root.parent / 'src').is_dir():
        project_root = project_root.parent
if not (project_root / 'src').is_dir():
    raise FileNotFoundError('Run this notebook from the repository root or notebooks directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.analysis.gtfs_eda import (
    build_eda_tables,
    load_processed_tables,
    top_connected_stops,
)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 30)

In [ ]:
gtfs_directory = project_root / 'data' / 'processed' / 'gtfs'
gtfs_tables = load_processed_tables(gtfs_directory)
eda = build_eda_tables(gtfs_tables)

print('Processed tables and row counts')
display(pd.DataFrame([{'table': name, 'rows': len(table)} for name, table in gtfs_tables.items()]))
print('Actual Parquet columns')
display(pd.DataFrame([{'table': name, 'columns': list(table.columns), 'dtypes': table.dtypes.astype(str).to_dict()} for name, table in gtfs_tables.items()]))

## Network summary and operator comparison

A trip count is the number of scheduled trip records in the feed. Stop-time count is the number of scheduled stop events. Calendar bounds and active weekdays come from recurring `calendar` records; they do not confirm that service operated on every date. Operator averages include all published routes, including routes with zero scheduled trips or no scheduled stops.

In [ ]:
display(eda['network_summary'])
display(eda['operator_comparison'])

## Route-level scheduled service

`scheduled_service_intensity` is the mean count of scheduled trip rows on weekdays when the route has service, derived by linking `trips.service_id` to the calendar weekday flags. It is a schedule descriptor, not an observed frequency of vehicles or passengers. Route names use only the observed route short/long name fields; route IDs remain available where names are absent.

In [ ]:
display(eda['routes'].head(20))
display(eda['route_statistics'])

## Stop connectivity

`routes_per_stop` counts scheduled route-stop relationships. `trips_per_stop` sums scheduled trip counts over those route-stop relationships. The lists below identify the most-connected and highest scheduled-service stops, not the busiest stops or stops with the most passengers. Stops present in the feed but absent from stop-time relationships are retained with zero service counts.

In [ ]:
display(eda['stop_connectivity'].head(20))
most_connected = pd.concat(
    [top_connected_stops(eda['stop_connectivity'].query('operator == @operator'), top_n=10)
     for operator in sorted(eda['stop_connectivity']['operator'].unique())],
    ignore_index=True,
)
highest_scheduled_service_stops = pd.concat(
    [top_connected_stops(eda['stop_connectivity'].query('operator == @operator'), top_n=10, metric='trips_per_stop')
     for operator in sorted(eda['stop_connectivity']['operator'].unique())],
    ignore_index=True,
)
display(most_connected)
display(highest_scheduled_service_stops)

## Calendar weekdays and service-day hours

Calendar weekday patterns are derived from `calendar` flags joined to trip service IDs. They are not observed operating-day counts, and the current feeds have no `calendar_dates` exception table.

The hourly profile counts scheduled stop departures (falling back to arrival where a departure time is blank). `service_day_hour` preserves extended GTFS values: 25:30:00 is service-day hour 25, wall-clock hour 01, on offset 1 (the following calendar day). Nothing at or above 24:00 is discarded.

In [ ]:
display(eda['weekday_service'])
display(eda['hourly_service'])
display(eda['peak_offpeak_service'])

In [ ]:
route_data = eda['routes'].copy()
operators = sorted(route_data['operator'].dropna().unique())
fig, axes = plt.subplots(1, max(len(operators), 1), figsize=(6 * max(len(operators), 1), 4), squeeze=False)
for axis, operator in zip(axes[0], operators):
    values = pd.to_numeric(route_data.loc[route_data['operator'].eq(operator), 'trip_count'], errors='coerce').dropna()
    sns.histplot(x=np.log10(values + 1), bins=30, ax=axis, color='#187b73')
    axis.set(title=operator, xlabel='log10(1 + scheduled trips per route)', ylabel='Route count')
fig.suptitle('Scheduled trips per route distribution')
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, max(len(operators), 1), figsize=(7 * max(len(operators), 1), 5), squeeze=False)
for axis, operator in zip(axes[0], operators):
    subset = route_data.loc[route_data['operator'].eq(operator)].nlargest(10, 'trip_count').copy()
    subset['chart_label'] = subset.get('route_name', subset['route_id']).fillna(subset['route_id'])
    sns.barplot(data=subset, x='trip_count', y='chart_label', ax=axis, color='#db8b32')
    axis.set(title=operator, xlabel='Scheduled trip records', ylabel='Observed route name or route ID')
fig.suptitle('Top routes by scheduled trips')
fig.tight_layout()
plt.show()

In [ ]:
stop_data = eda['stop_connectivity']
fig, axes = plt.subplots(1, max(len(operators), 1), figsize=(7 * max(len(operators), 1), 5), squeeze=False)
for axis, operator in zip(axes[0], operators):
    subset = top_connected_stops(stop_data.loc[stop_data['operator'].eq(operator)], top_n=10).copy()
    subset['chart_label'] = subset['stop_name'].fillna(subset['stop_id']) if 'stop_name' in subset else subset['stop_id']
    sns.barplot(data=subset, x='routes_per_stop', y='chart_label', ax=axis, color='#3974a5')
    axis.set(title=operator, xlabel='Scheduled routes serving stop', ylabel='Observed stop name or stop ID')
fig.suptitle('Most-connected stops by route connectivity')
fig.tight_layout()
plt.show()

In [ ]:
hourly = eda['hourly_service']
fig, axes = plt.subplots(1, max(len(operators), 1), figsize=(7 * max(len(operators), 1), 4), squeeze=False)
for axis, operator in zip(axes[0], operators):
    subset = hourly.loc[hourly['operator'].eq(operator)]
    sns.lineplot(data=subset, x='service_day_hour', y='scheduled_departure_count', marker='o', ax=axis, color='#a3485a')
    axis.axvline(24, color='black', linestyle='--', linewidth=1)
    axis.set(title=operator, xlabel='Service-day hour (hours >= 24 retained)', ylabel='Scheduled stop departures')
fig.suptitle('Hourly scheduled service profile')
fig.tight_layout()
plt.show()

In [ ]:
comparison = eda['operator_comparison']
comparison_metrics = [
    ('routes', 'Route records'), ('stops', 'Stop records'), ('trips', 'Scheduled trip records'),
    ('stop_times', 'Scheduled stop-time records'), ('average_trips_per_route', 'Mean scheduled trips per route'),
    ('average_stops_per_route', 'Mean scheduled stops per route'),
]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for axis, (metric, label) in zip(axes.flat, comparison_metrics):
    sns.barplot(data=comparison, x='operator', y=metric, ax=axis, color='#4b8c62')
    axis.set(title=label, xlabel='Operator/feed', ylabel=label)
fig.suptitle('Feed-level schedule and network comparison (separate scales by metric)')
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=route_data, x='operator', y='trip_count', showfliers=False, ax=axes[0], color='#89b7ae')
axes[0].set(title='Scheduled trip records by route', xlabel='Operator/feed', ylabel='Scheduled trips per route')
sns.boxplot(data=route_data, x='operator', y='stop_count', showfliers=False, ax=axes[1], color='#e2ae6f')
axes[1].set(title='Scheduled route coverage', xlabel='Operator/feed', ylabel='Distinct scheduled stops per route')
fig.suptitle('Route/service distributions (boxplot fliers omitted; see summary table)')
fig.tight_layout()
plt.show()

## Findings and interpretation limits

The next cell prints feed-specific values from the processed files at execution time; it does not hardcode results. Interpret these only as published schedule/network structure.

In [ ]:
for _, row in eda['network_summary'].iterrows():
    operator = row['operator']
    route_rows = route_data.loc[route_data['operator'].eq(operator)]
    stop_rows = stop_data.loc[stop_data['operator'].eq(operator)]
    top_route = route_rows.sort_values(['trip_count', 'route_id'], ascending=[False, True]).head(1)
    top_stop = top_connected_stops(stop_rows, top_n=1)
    route_label = (top_route.iloc[0].get('route_name') or top_route.iloc[0]['route_id']) if not top_route.empty else 'not available'
    stop_label = (top_stop.iloc[0].get('stop_name') or top_stop.iloc[0]['stop_id']) if not top_stop.empty else 'not available'
    print(f"{operator}: {int(row['routes']):,} routes, {int(row['stops']):,} stops, {int(row['trips']):,} scheduled trip records, and {int(row['stop_times']):,} scheduled stop-time records.")
    print(f"  Published calendar span: {row['service_start']} through {row['service_end']}; active weekdays: {row['active_weekdays']}.")
    print(f"  Highest scheduled-trip route: {route_label} ({int(top_route.iloc[0]['trip_count']):,} trip records) when available.") if not top_route.empty else None
    print(f"  Most-connected stop: {stop_label} ({int(top_stop.iloc[0]['routes_per_stop'])} routes) when available.") if not top_stop.empty else None

print('No passenger-demand, ridership, actual-usage, or passenger-volume findings are supported by these GTFS tables.')

### Limitations

- GTFS represents scheduled service; it does not establish whether trips ran or how many people used them.
- Service intensity and scheduled trip frequency are not ridership or passenger demand. Stop connectivity is not passenger volume.
- Feed coverage is limited to the service-date bounds shown in the calendar summary. These dates are not historical observations of service delivery.
- Neither feed has `calendar_dates`; ad hoc service exceptions cannot be represented in the weekday counts.
- TGSRTC publishes routes with no trip records, and HMRL publishes stops without stop-time references. Their causes are unknown; they are retained rather than repaired.
- Operator metrics have compatible mechanical definitions but different modes, route concepts, feed coverage, and schedules. They should not be interpreted as a performance or demand ranking.
- Extended GTFS service-day times are preserved. A service-day hour above 23 is converted to a next-day wall-clock hour only in the separate `clock_hour` field.
- No passenger data was acquired or fabricated. No forecasting or advanced machine learning is performed in Phase 3A.